In [1]:
import os
import tensorflow as tf

os.environ['CUDA_VISIBLE_DEVICES'] = '5'
gpus = tf.config.list_physical_devices('GPU')
tf.config.experimental.set_memory_growth(gpus[0], True)
tf.config.experimental.set_virtual_device_configuration(
    gpus[0],
    [tf.config.experimental.VirtualDeviceConfiguration(memory_limit=1024)]
)

I0000 00:00:1775604606.046335 4180900 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
import torch

BASE_PATH = "../"
device = "cuda" if torch.cuda.is_available() else "cpu"

HF_TOKEN = ...

In [ ]:
import pandas as pd
import glob
import os
from datasets import Dataset

BASE_PATH = "../"
CLEANING_DATASET = BASE_PATH + "data/cleaning/"
MODEL_SAVE_PATH = BASE_PATH + "models/rut5-cleaner/"
TUNED_MODEL_SAVE_PATH = BASE_PATH + "models/rut5-cleaner-tuned/"

csv_files = sorted(glob.glob(CLEANING_DATASET + "*.csv"))
dfs = [pd.read_csv(f) for f in csv_files]
full_df = pd.concat(dfs, ignore_index=True)

full_df = full_df.dropna().reset_index(drop=True)

dataset = Dataset.from_pandas(full_df)
dataset = dataset.train_test_split(test_size=0.1)

print(f"Train size: {len(dataset['train'])}")
print(f"Test size: {len(dataset['test'])}")

/data/koposovti/lecture-transcriber/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train size: 606
Test size: 68


In [4]:
from transformers import AutoTokenizer

model_checkpoint = "cointegrated/rut5-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, token=HF_TOKEN)

max_input_length = 512
max_target_length = 512

def preprocess_function(examples):
    inputs = ["clean: " + doc for doc in examples["raw_text"]]
    model_inputs = tokenizer(
        inputs, 
        max_length=512, 
        truncation=True, 
        padding="max_length"
    )

    labels = tokenizer(
        text_target=examples["target_text"], 
        max_length=512, 
        truncation=True, 
        padding="max_length"
    )

    labels_with_ignore_index = []
    for labels_example in labels["input_ids"]:
        labels_example = [
            (l if l != tokenizer.pad_token_id else -100) 
            for l in labels_example
        ]
        labels_with_ignore_index.append(labels_example)

    model_inputs["labels"] = labels_with_ignore_index
    return model_inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/data/koposovti/lecture-transcriber/.venv/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Map: 100%

In [5]:
def check_token_lengths(dataset, tokenizer, max_val=512):
    lengths = [len(tokenizer.encode(x)) for x in dataset['raw_text']]
    oversized = [l for l in lengths if l > max_val]
    
    print(f"Avg length: {sum(lengths)/len(lengths):.1f} tokens")
    print(f"Max length: {max(lengths)} tokens")
    print(f"Percentage of cutted raws: {len(oversized)/len(lengths)*100:.2f}%")

check_token_lengths(full_df, tokenizer)

Avg length: 331.4 tokens
Max length: 557 tokens
Percentage of cutted raws: 7.27%


In [6]:
torch.cuda.empty_cache()

In [7]:
import torch
from transformers import AutoConfig, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

model_checkpoint = "cointegrated/rut5-base"

config = AutoConfig.from_pretrained(model_checkpoint, token=HF_TOKEN)
config.tie_word_embeddings = False 
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint, config=config, token=HF_TOKEN)

model.encoder.embed_tokens.weight = model.shared.weight
model.decoder.embed_tokens.weight = model.shared.weight

# model.gradient_checkpointing_enable()


In [9]:
training_args = Seq2SeqTrainingArguments(
    output_dir=MODEL_SAVE_PATH,
    eval_strategy="epoch", 
    save_strategy="epoch",

    learning_rate=5e-4,   
    num_train_epochs=10, 
    per_device_train_batch_size=1, 
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16, 
    
    weight_decay=0.01,
    save_total_limit=2,
    
    bf16=True,
    optim="adafactor",
    lr_scheduler_type="cosine",
    warmup_steps=80,
    
    predict_with_generate=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=10,
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

torch.cuda.empty_cache()

trainer.train()

Epoch,Training Loss,Validation Loss
1,9.775500,2.886389
2,2.899000,2.159949
3,2.404100,2.015320
4,2.100100,1.916658
5,1.802900,1.896953
6,1.672700,1.869476
7,1.457000,1.855631
8,1.305300,1.898732
9,1.151600,1.914882
10,1.032200,1.948934


KeyboardInterrupt: 

In [10]:
model.save_pretrained(TUNED_MODEL_SAVE_PATH)
tokenizer.save_pretrained(TUNED_MODEL_SAVE_PATH)

('../models/rut5-cleaner-tuned/tokenizer_config.json',
 '../models/rut5-cleaner-tuned/special_tokens_map.json',
 '../models/rut5-cleaner-tuned/spiece.model',
 '../models/rut5-cleaner-tuned/added_tokens.json',
 '../models/rut5-cleaner-tuned/tokenizer.json')

In [11]:
from transformers import AutoConfig, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained(TUNED_MODEL_SAVE_PATH)
config = AutoConfig.from_pretrained(TUNED_MODEL_SAVE_PATH)
config.tie_word_embeddings = False 
model = AutoModelForSeq2SeqLM.from_pretrained(TUNED_MODEL_SAVE_PATH, config=config)
model.encoder.embed_tokens.weight = model.shared.weight
model.decoder.embed_tokens.weight = model.shared.weight

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers
/data/koposovti/lecture-transcriber/.venv/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [12]:
import time

def clean_text(text, model, tokenizer):
    model.eval()
    device = model.device
    input_text = "clean: " + text
    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=512,
            num_beams=5,  
            repetition_penalty=2.5,
            length_penalty=1.0,
            early_stopping=True
        )
    
    return  tokenizer.decode(outputs[0], skip_special_tokens=True)

raw_text = dataset['train'][0]['raw_text']
print(raw_text)
print(clean_text(raw_text, model, tokenizer))

Она уже записана, и он её не может записать. И дальше они просто так в цикле крутятся. У них нет прогресса никакого, ни у одного из них. Они просто крутятся в цикле бесконечно. Но они не заблокированы. Они не спят, они не ждут. Они в цикле крутятся. Постоянно что-то делают. А может, получается, что в какой-то момент какой-то из них кто-нибудь схватит два флага? Да, да, да. Лайвлоки — это штука, она обычно недолго живущая. То есть это типа не то, что у вас программа на пять лет затормозила, и всё, её надо короче келять и перезапускать. Нет, возможно, что вот такая ситуация, она продлится, я не знаю, пять секунд, потом они один окажется всё-таки быстрее, шедулер, что-нибудь сделает свою магию, и у вас этот лок уйдёт. А если счётчик добавить? Можно сказать, что будет?
Она уже записана, и её нельзя записать повторно. Они продолжают крутиться в цикле без прогресса. Они не спят, не ждут — постоянно выполняют операции. В какой-то момент кто-то захватит два флага? Лайвлоки — это непредсказуема